In [13]:
# Import the required libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


In [14]:
# Load the training and test feature matrices
X_train = pd.read_parquet('../data/X_train.parquet', engine='fastparquet')
X_test = pd.read_parquet('../data/X_test.parquet', engine='fastparquet')

y_train = pd.read_parquet('../data/y_train.parquet', engine='fastparquet')['is_fraud']
y_test = pd.read_parquet('../data/y_test.parquet', engine='fastparquet')['is_fraud']
    
# Verify the dimensions of the datasets
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

X_train shape: (1296675, 19) | X_test shape: (555719, 19)


In [15]:
# Scale the features because Logistic Regression is sensitive to different numerical scales
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
# Initialize Logistic Regression using 'class_weight=balanced' to handle class imbalance
logic_model = LogisticRegression(class_weight='balanced', max_iter=1000)

# Train the baseline model
logic_model.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [17]:
# Generamos predicciones en el conjunto de prueba
y_pred = logic_model.predict(X_test_scaled)

# Display the confusion matrix and classification report
print("=== CONFUSION MATRIX (LOGISTIC REGRESSION) ===")
print(confusion_matrix(y_test, y_pred))
print("\n=== CLASSIFICATION REPORT (LOGISTIC REGRESSION) ===")
print(classification_report(y_test, y_pred))

=== CONFUSION MATRIX (LOGISTIC REGRESSION) ===
[[486296  67278]
 [   556   1589]]

=== CLASSIFICATION REPORT (LOGISTIC REGRESSION) ===
              precision    recall  f1-score   support

           0       1.00      0.88      0.93    553574
           1       0.02      0.74      0.04      2145

    accuracy                           0.88    555719
   macro avg       0.51      0.81      0.49    555719
weighted avg       1.00      0.88      0.93    555719



>**Logistic Regression Baseline Evaluation:** Precision and F1-Score for the positive class (fraud) proved insufficient for production needs. Switching strategy to **XGBoost** to capture non-linear decision boundaries.

In [18]:
# Calculate the class weighting factor for imbalanced classes (Negative Cases / Positive Cases)
class_counts = y_train.value_counts()
scale_weight = class_counts[0] / class_counts[1]

# Initialize XGBoost with class balancing configuration
xgb_model = XGBClassifier(
    n_estimators=300, 
    max_depth=5, 
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)

# Train XGBoost using the original features (feature scaling is not required)
xgb_model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [19]:
# Generate standard predictions using the default threshold (0.50)
y_pred_xgb = xgb_model.predict(X_test)

print("=== CONFUSION MATRIX (XGBOOST - THRESHOLD 0.50) ===")
print(confusion_matrix(y_test, y_pred_xgb))
print("\n=== CLASSIFICATION REPORT (XGBOOST - THRESHOLD 0.50) ===")
print(classification_report(y_test, y_pred_xgb))

=== CONFUSION MATRIX (XGBOOST - THRESHOLD 0.50) ===
[[550115   3459]
 [   146   1999]]

=== CLASSIFICATION REPORT (XGBOOST - THRESHOLD 0.50) ===
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    553574
           1       0.37      0.93      0.53      2145

    accuracy                           0.99    555719
   macro avg       0.68      0.96      0.76    555719
weighted avg       1.00      0.99      0.99    555719



In [20]:
# Evaluate predictions on the training dataset to measure potential overfitting
y_pred_train = xgb_model.predict(X_train)

print("=== TRAINING SET REPORT (OVERFITTING EVALUATION) ===")
print(classification_report(y_train, y_pred_train))

=== TRAINING SET REPORT (OVERFITTING EVALUATION) ===
              precision    recall  f1-score   support

           0       1.00      0.99      1.00   1289169
           1       0.49      1.00      0.66      7506

    accuracy                           0.99   1296675
   macro avg       0.75      1.00      0.83   1296675
weighted avg       1.00      0.99      1.00   1296675



In [21]:
# Obtain continuous fraud probabilities (class 1) instead of fixed binary predictions
y_probs = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate using a 0.60 threshold to reduce false positives
y_pred_60 = (y_probs >= 0.60).astype(int)

print("=== THRESHOLD 0.60 ===")
print(confusion_matrix(y_test, y_pred_60))
print(classification_report(y_test, y_pred_60))

=== THRESHOLD 0.60 ===
[[550708   2866]
 [   168   1977]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    553574
           1       0.41      0.92      0.57      2145

    accuracy                           0.99    555719
   macro avg       0.70      0.96      0.78    555719
weighted avg       1.00      0.99      1.00    555719



In [22]:
# Evaluate using a 0.70 threshold to optimize precision while maintaining high recall
y_pred_70 = (y_probs >= 0.70).astype(int)

print("=== THRESHOLD 0.70 ===")
print(confusion_matrix(y_test, y_pred_70))
print(classification_report(y_test, y_pred_70))

=== THRESHOLD 0.70 ===
[[551240   2334]
 [   196   1949]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.46      0.91      0.61      2145

    accuracy                           1.00    555719
   macro avg       0.73      0.95      0.80    555719
weighted avg       1.00      1.00      1.00    555719



In [23]:
# The best performance was achieved with a 0.70 threshold.
# Therefore, cross-validation will preserve this threshold when calculating the F1-score.
# We also evaluate the model independently of the threshold using Average Precision.

# Store performance metrics for each of the 5 validation folds
pr_auc_scores = []
f1_scores_70 = [] 

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in skf.split(X_train, y_train):

# Store performance metrics for each of the 5 validation folds
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train XGBoost on the current fold
    xgb_model.fit(X_tr, y_tr)

    # Obtain fraud probabilities on the validation set
    probs = xgb_model.predict_proba(X_val)[:, 1]

    # Calculate Precision-Recall AUC (independent of the classification threshold)
    pr_auc_scores.append(average_precision_score(y_val, probs))

    # Convert probabilities into binary predictions using the optimized 0.70 threshold
    preds_70 = (probs >= 0.70).astype(int)

    # Calculate F1-score on the validation fold
    score = f1_score(y_val, preds_70)
    f1_scores_70.append(score)

# Display the mean and standard deviation of the cross-validation metrics
print("=== FINAL CROSS-VALIDATION RESULTS (5-FOLD) ===")
print("Mean PR-AUC:", np.mean(pr_auc_scores), "±", np.std(pr_auc_scores))
print("Mean F1-Score (Threshold = 0.70):", np.mean(f1_scores_70), "±", np.std(f1_scores_70))

=== FINAL CROSS-VALIDATION RESULTS (5-FOLD) ===
Mean PR-AUC: 0.9124875025823698 ± 0.002790659447532877
Mean F1-Score (Threshold = 0.70): 0.7123345774051133 ± 0.004758140919050962


### Model Justification & Decision Strategy

1. **Logistic Regression Baseline:**  
   Used as a benchmark model. Although `class_weight='balanced'` improved fraud detection, the model generated an unacceptable number of false positives (low precision), which in a production environment could unnecessarily block legitimate customer transactions.

2. **XGBoost & Gradient Boosting Advantages:**  
   XGBoost captures complex non-linear relationships between features, such as combinations of transaction time, geographic distance, and transaction amount. The `scale_pos_weight` parameter directly addresses the severe class imbalance.

3. **Decision Threshold Optimization:**  
   Standard classifiers use a default probability threshold of 0.50. Increasing this threshold to **0.70** reduces false alarms while maintaining strong recall for fraudulent transactions.

4. **Stratified K-Fold Cross-Validation:**  
   Due to the extreme class imbalance (~0.52% positive cases), standard cross-validation may produce inconsistent class distributions across folds. `StratifiedKFold` with 5 splits ensures reliable evaluation using **PR-AUC** and **F1-Score** metrics.